## Setup

In [1]:
import os
import re
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path


import sys, platform
print("=== Environment Info ===")
print(f"Python envi    : {sys.executable}")
print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"geopandas      : {gpd.__version__}")
print(f"numpy          : {np.__version__}")
print("========================")




# Get and print current working directory
cwd = os.path.dirname(os.getcwd())
cwd = Path(cwd)
print("Current working directory:", cwd)

# Define output path
dir_aoi    = cwd / "data"/ "aoi"
dir_share  = Path(r"G:\Shared drives\invest-health\City500")
dir_figure = cwd / "figures"


=== Environment Info ===
Python envi    : c:\Users\pc\.conda\envs\geo_env\python.exe
Python version : 3.11.13
Platform       : Windows-10-10.0.22631-SP0
geopandas      : 0.14.4
numpy          : 2.2.6
Current working directory: d:\natcap\invest-mental-health


## Data

In [4]:
## Load data 1 with identified cities short list 

aoi_f = dir_aoi / 'places_in_or_touching_metros_487.shp'

aoi500 = gpd.read_file(aoi_f)

print(aoi500.columns.to_list())

print(aoi500.shape)

['STATEFP', 'PLACEFP', 'PLACENS', 'AFFGEOID_l', 'GEOID_PLAC', 'NAME_PLACE', 'NAMELSAD_l', 'STUSPS', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'SUMLEV', 'STATE', 'PLACE', 'POP2020_le', 'POP2021_le', 'POP2022_le', 'POP2023_le', 'POP2024_le', 'POP_KM2_le', 'CSAFP', 'CBSAFP', 'METDIVFP', 'AFFGEOID_r', 'GEOID_METR', 'NAME_METRO', 'NAMELSAD_r', 'LSAD_x', 'MDIV', 'LSAD_y', 'SBASE2020', 'POP2020_ri', 'POP2021_ri', 'POP2022_ri', 'POP2023_ri', 'POP2024_ri', 'GEOID_MDIV', 'POP_KM2_ri', 'count', 'level', 'AFFGEOID', 'NAMELSAD', 'POP2020', 'POP2021', 'POP2022', 'POP2023', 'POP2024', 'POP_KM2', 'geometry']
(487, 50)


In [ ]:
## Load data 2 with tracts

f = dir_aoi / 'tract_place_2020.shp'
tract_place = gpd.read_file(f)

print(tract_place.columns.to_list())

['GEOID', 'NAME_PLACE', 'STATEFP_1', 'COUNTYFP', 'ALAND_1', 'variable', 'pop_tract', 'year', 'STATEFP_2', 'PLACEFP', 'PLACENS', 'GEOID_PLAC', 'STUSPS', 'STATE_NAME', 'LSAD', 'ALAND_2', 'AWATER', 'SUMLEV', 'STATE', 'PLACE', 'POP2020', 'POP2021', 'POP2022', 'POP2023', 'POP2024', 'POP_KM2', 'overlap_ra', 'geometry']


### get the centroids of each city 


Summary Comparison Table

| Method | Guaranteed Inside? | Visually Centered? | Speed | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Centroid** | ❌ No (can fall outside) | ✅ Yes (Center of Mass) | ⚡ Fast | Mathematical analysis, simple shapes. |
| **Representative Point** | ✅ Yes | ❌ No (Arbitrary) | 🚀 Fastest | Spatial joins, ensuring point is strictly on the feature. |
| **Polylabel** | ✅ Yes | ✅ Yes (Visual Center) | 🐢 Slow | Cartography, labeling irregular shapes, finding the "deepest" point. |


In [8]:
import geopandas as gpd
from shapely.ops import polylabel
from pathlib import Path

# --- SETUP VARIABLES (Assuming these exist in your environment) ---
# dir_aoi = Path("path/to/your/data")
# aoi500 = gpd.read_file(...) 

# 1) Work in an equal-area projected CRS (EPSG:5070 = US Albers)
# Note: Ensure aoi500 exists. We copy it to avoid modifying the original variable.
gdf = aoi500.copy().to_crs(5070)

# 2) Plain centroid
gdf["centroid_geom"] = gdf.geometry.centroid

# 3) Guaranteed-inside point (Representative Point)
gdf["inside_point"] = gdf.geometry.representative_point()

# 4) Polylabel
# Added try/except to lambda to handle potential empty geometries gracefully
def safe_polylabel(geom, tol=5):
    try:
        return polylabel(geom, tolerance=tol)
    except:
        return None

gdf["polylabel_point"] = gdf.geometry.apply(safe_polylabel)

# 5) Export Loop (Reduces code repetition)
export_config = {
    "centroid": ("centroid_geom", "aoi487_centroids.gpkg"),
    "point_on_surface": ("inside_point", "aoi487_inside.gpkg"),
    "polylabel": ("polylabel_point", "aoi487_polylabel.gpkg")
}

for layer_name, (col_name, filename) in export_config.items():
    # Create a clean subset with attributes, keeping only the specific geometry
    out_gdf = gdf.copy()
    out_gdf = out_gdf.set_geometry(col_name)
    
    # Drop the other temporary geometry columns to keep file size down
    cols_to_drop = ["centroid_geom", "inside_point", "polylabel_point", "geometry"]
    # Only drop the ones that exist and aren't the current active geometry
    cols_to_drop = [c for c in cols_to_drop if c in out_gdf.columns and c != col_name]
    out_gdf = out_gdf.drop(columns=cols_to_drop)
    
    # Ensure CRS is preserved
    out_gdf.crs = gdf.crs
    
    # Save
    out_path = dir_aoi / filename
    out_gdf.to_file(out_path, layer=layer_name, driver="GPKG")
    print(f"Saved {layer_name} to {out_path}")

Saved centroid to d:\natcap\invest-mental-health\data\aoi\aoi487_centroids.gpkg
Saved point_on_surface to d:\natcap\invest-mental-health\data\aoi\aoi487_inside.gpkg
Saved polylabel to d:\natcap\invest-mental-health\data\aoi\aoi487_polylabel.gpkg


In [5]:
## keep only rows in data2 whose IDs appear in data1.

# Cast to string to avoid 06085→6085 issues
aoi500["GEOID_PLAC"] = aoi500["GEOID_PLAC"].astype(str)
tract_place["GEOID_PLAC"] = tract_place["GEOID_PLAC"].astype(str)

# 2) Get unique IDs from data1
ids = aoi500["GEOID_PLAC"].dropna().unique()
print(f'There are {len(ids)} places/cities included.')

# 3) Filter data2 (two equivalent options)
# Option A: isin (fast & simple)
tract_place_filtered = tract_place[tract_place["GEOID_PLAC"].isin(ids)].copy()

There are 487 places/cities included.


### save each city as a seperate file

In [ ]:
## 1. save the shp with city bou only

dir_aoi_each  = dir_share / 'aoi_each'   # output folder
dir_aoi_each.mkdir(parents=True, exist_ok=True)

id_col = "GEOID_PLAC" # column to split by

def safe_name(s, maxlen=50):
    """Make a safe filename from an id value."""
    s = str(s)
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:maxlen] or "NA"

for geoid, g in aoi500.groupby(id_col):
    fname = f"city_{safe_name(geoid)}.shp"
    g.to_file(dir_aoi_each / fname, driver="ESRI Shapefile")
    # print("saved:", dir_aoi_each / fname)    


In [8]:
## 2. save the shp with both city and tract 

dir_aoi_each  = dir_share / 'aoi_each_with_tract'   # output folder
dir_aoi_each.mkdir(parents=True, exist_ok=True)

id_col = "GEOID_PLAC" # column to split by

def safe_name(s, maxlen=50):
    """Make a safe filename from an id value."""
    s = str(s)
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:maxlen] or "NA"

for geoid, g in tract_place_filtered.groupby(id_col):
    fname = f"city_{safe_name(geoid)}_tract.shp"
    g.to_file(dir_aoi_each / fname, driver="ESRI Shapefile")
    # print("saved:", dir_aoi_each / fname)    
